# Week 8 — PhaseNet output verification
Same checks as Week 4's EQTransformer verification: per-station totals,
sanity ranges, and the 4 September mainshock temporal spike - now also
compared directly against EQTransformer's own numbers as a cross-check
between the two pickers.

In [ ]:
import os, glob
import pandas as pd

PROJECT_DIR = os.path.expanduser("~/tvz_project")
PICKS_DIR = os.path.join(PROJECT_DIR, "picks", "phasenet")

picks_files = sorted(glob.glob(os.path.join(PICKS_DIR, "*_thresh0.3_picks.csv")))
print(f"{len(picks_files)} station pick files found at threshold 0.3")

In [ ]:
picks_df = pd.concat([pd.read_csv(f) for f in picks_files], ignore_index=True)
picks_df["station_code"] = picks_df["station"].str.split(".").str[1]
picks_df["time"] = pd.to_datetime(picks_df["time"])

print(f"Total picks: {len(picks_df)}")
print(f"P picks: {(picks_df['phase'] == 'P').sum()}, S picks: {(picks_df['phase'] == 'S').sum()}")
print(f"Probability range: {picks_df['probability'].min():.4f} - {picks_df['probability'].max():.4f}")
print(f"Mean picks/station: {len(picks_df) / len(picks_files):.0f}")
print(f"\nFor comparison, EQTransformer (Week 4): 339,857 total picks, 5,664 picks/station average")

In [ ]:
# Per-station summary, same sanity checks as Week 4
summary = picks_df.groupby("station_code").agg(
    n_picks=("time", "count"),
    n_P=("phase", lambda x: (x == "P").sum()),
    n_S=("phase", lambda x: (x == "S").sum()),
    mean_probability=("probability", "mean"),
).reset_index().sort_values("n_picks")

pd.set_option("display.max_rows", None)
print(f"Stations with 0 picks: {(summary['n_picks'] == 0).sum()}")
print(f"Mean picks/station: {summary['n_picks'].mean():.0f}, median: {summary['n_picks'].median():.0f}")
print(f"\nLowest 5 stations:")
print(summary.head(5).to_string(index=False))
print(f"\nHighest 5 stations:")
print(summary.tail(5).to_string(index=False))

out_csv = os.path.join(PROJECT_DIR, "picks", "phasenet_summary.csv")
summary.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
# The critical check: does the 4 Sept 2019 mainshock show up as a spike,
# same as it did for EQTransformer (3.1x the daily mean, Week 4)?
import datetime

picks_df["date"] = picks_df["time"].dt.date
daily_counts = picks_df.groupby("date").size()

mainshock_date = datetime.date(2019, 9, 4)
window = daily_counts.loc[datetime.date(2019, 8, 30):datetime.date(2019, 9, 10)]
print("Daily pick counts, 30 Aug - 10 Sep 2019:")
print(window)

print(f"\nOverall daily mean: {daily_counts.mean():.0f}")
print(f"Mainshock day (4 Sep) count: {daily_counts.get(mainshock_date, 0):.0f}")
print(f"Ratio to daily mean: {daily_counts.get(mainshock_date, 0) / daily_counts.mean():.1f}x")
print(f"(EQTransformer's equivalent ratio, Week 4, was 3.1x)")